# Automobile Price Prediction

End-to-end regression project on the *Automobile* (imports-85) dataset: EDA, Pearson correlation with p-values, simple / multiple / polynomial regression, pipelines, train-test split, cross-validation, Ridge regularization and Grid Search.


## IBM PRACTICE 2

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
try:
    from pyodide.http import pyfetch
except ModuleNotFoundError:
    import urllib.request

    class _Response:
        def __init__(self, status, data):
            self.status = status
            self._data = data

        async def bytes(self):
            return self._data

    async def pyfetch(url):
        with urllib.request.urlopen(url) as response:
            return _Response(response.status, response.read())

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())
            

In [ ]:
file_path= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/automobileEDA.csv"

In [ ]:
await download(file_path, "usedcars.csv")
file_name="usedcars.csv"

In [ ]:
df = pd.read_csv(file_name, header=0)

In [ ]:
df.describe(include="all").T

In [ ]:
df.head()

In [ ]:
df["price"].max()
df["make"].value_counts()
print(f"Maximum price: {df['price'].max()}")
print(f"Most common make: {df['make'].value_counts().idxmax()} with {df['make'].value_counts().max()} occurrences")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

In [ ]:
print(df.dtypes)

In [ ]:
df['peak-rpm'].dtypes

In [ ]:
numeric_df = df.select_dtypes(include=['float64', 'int64'])
numeric_df.corr()

In [ ]:
df[['bore', 'stroke', 'compression-ratio', 'horsepower']].corr()

## POSITIVE CORRELATION

In [ ]:
# Engine size as potential predictor variable of price
sns.regplot(x="engine-size", y="price", data=df)
plt.ylim(0,)

In [ ]:
df[["engine-size", "price"]].corr()

## NEGATIVE CORRELATION

In [ ]:
sns.regplot(x="highway-mpg", y="price", data=df)

In [ ]:
df[["highway-mpg", "price"]].corr()

## WEAK LINEAR RELATIONSHIP

In [ ]:
sns.regplot(x="peak-rpm", y="price", data=df)

In [ ]:
df[['peak-rpm','price']].corr()

In [ ]:
df[['stroke','price']].corr()

In [ ]:
sns.regplot(x="stroke", y="price", data=df)

## CATEGORIAL VARIABLES 

In [ ]:
sns.boxplot(x="body-style", y="price", data=df)

In [ ]:
sns.boxplot(x="engine-location", y="price", data=df)

In [ ]:
sns.boxplot(x="drive-wheels", y="price", data=df)

## Descriptive Statistical Analysis

In [ ]:
df.describe(include=['object'])

In [ ]:
df['drive-wheels'].value_counts()

In [ ]:
drive_wheels_counts = df['drive-wheels'].value_counts().to_frame()
drive_wheels_counts.reset_index(inplace=True)
drive_wheels_counts=drive_wheels_counts.rename(columns={'drive-wheels': 'value_counts'})
drive_wheels_counts

In [ ]:
drive_wheels_counts.index.name = 'drive-wheels'
drive_wheels_counts

In [ ]:
# engine-location as variable
engine_loc_counts = df['engine-location'].value_counts().to_frame()
engine_loc_counts.rename(columns={'engine-location': 'value_counts'}, inplace=True)
engine_loc_counts.index.name = 'engine-location'
engine_loc_counts.head(10)

## BASICS OF GROUPING 

In [ ]:
df['drive-wheels'].unique()

In [ ]:
df_group_one = df[['drive-wheels','body-style','price']]

In [ ]:
# grouping results
df_grouped = df_group_one.groupby(['drive-wheels'], as_index=False).agg({'price': 'mean'})
df_grouped

In [ ]:
df_gptest = df[['drive-wheels','body-style','price']]
grouped_test1 = df_gptest.groupby(['drive-wheels','body-style'],as_index=False).mean()
grouped_test1

In [ ]:
grouped_pivot = grouped_test1.pivot(index='drive-wheels',columns='body-style')
grouped_pivot

In [ ]:
grouped_pivot = grouped_pivot.fillna(0) #fill missing values with 0
grouped_pivot

In [ ]:
# grouping results
df_gptest2 = df[['body-style','price']]
grouped_test_bodystyle = df_gptest2.groupby(['body-style'],as_index= False).mean()
grouped_test_bodystyle

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

In [ ]:
plt.pcolor(grouped_pivot, cmap='RdBu')
plt.colorbar()
plt.show()

Yes — if the “heatmap above” is the correlation heatmap from your notebook, it is showing how strongly pairs of variables are related to each other.

What it is:
- A heatmap is a grid of colored cells.
- Each cell represents the correlation between two variables.
- In your notebook, this is created with:
    - `C = df.corr(numeric_only=True)`
    - `sns.heatmap(C, annot=True, cmap="RdBu_r", vmin=-1, vmax=1, center=0)`

How to read it:
- 1.0 = perfect positive correlation
    - As one variable increases, the other also increases
    - Example: larger engine size and higher price often go together
- -1.0 = perfect negative correlation
    - As one variable increases, the other decreases
    - Example: higher fuel efficiency and lower price can sometimes be related
- 0.0 = no linear relationship
- The diagonal cells are always 1.0
    - because each variable is perfectly correlated with itself

Color meaning:
- In your heatmap, warm colors (red/orange) usually mean positive correlation
- Cool colors (blue) usually mean negative correlation
- Near white/neutral colors mean weak or no correlation

Why it matters:
- It helps you quickly see which features are related
- This is useful before building models, because highly correlated features may carry similar information
- For example:
    - If `engine-size` and `price` are both strongly red, that means larger engines tend to be more expensive
    - If `highway-mpg` and `price` are blue, it may suggest more efficient cars tend to be cheaper

Important note:
- Correlation only measures linear relationships
- It does not mean one variable causes the other
- Two variables can be strongly related without being causal

So in short:
- The heatmap is a visual summary of the correlation matrix
- Darker red = stronger positive relationship
- Darker blue = stronger negative relationship
- Light/neutral = weak or no relationship

If you want, I can also explain the heatmap in your notebook step by step using the actual variables shown there, like `price`, `engine-size`, `horsepower`, `city-mpg`, etc.

In [ ]:
fig, ax = plt.subplots()
im = ax.pcolor(grouped_pivot, cmap='RdBu')

#label names
row_labels = grouped_pivot.columns.levels[1]
col_labels = grouped_pivot.index

#move ticks and labels to the center
ax.set_xticks(np.arange(grouped_pivot.shape[1]) + 0.5, minor=False)
ax.set_yticks(np.arange(grouped_pivot.shape[0]) + 0.5, minor=False)

#insert labels
ax.set_xticklabels(row_labels, minor=False)
ax.set_yticklabels(col_labels, minor=False)

#rotate label if too long
plt.xticks(rotation=90)

fig.colorbar(im)
plt.show()

## CORRELATION AND CAUSATION

Correlation: a measure of the extent of interdependence between variables.

Causation: the relationship between cause and effect between two variables.

It is important to know the difference between these two. Correlation does not imply causation. Determining correlation is much simpler the determining causation as causation may require independent experimentation.

## Pearson Correlation 

The Pearson Correlation measures the linear dependence between two variables X and Y.

The resulting coefficient is a value between -1 and 1 inclusive, where:

1: Perfect positive linear correlation.
0: No linear correlation, the two variables most likely do not affect each other.
-1: Perfect negative linear correlation.

In [ ]:
df.select_dtypes(include=['number']).corr()

## P-value

What is this P-value? The P-value is the probability value that the correlation between these two variables is statistically significant. Normally, we choose a significance level of 0.05, which means that we are 95% confident that the correlation between the variables is significant.

By convention, when the

p-value is 
 0.001: we say there is strong evidence that the correlation is significant.
the p-value is 
 0.05: there is moderate evidence that the correlation is significant.
the p-value is 
 0.1: there is weak evidence that the correlation is significant.
the p-value is 
 0.1: there is no evidence that the correlation is significant

In [ ]:
from scipy import stats

## Wheel-Base vs. Price 

In [ ]:
pearson_coef, p_value = stats.pearsonr(df['wheel-base'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P =", p_value)  

## HORSEPOWER VS PRICE 

In [ ]:
pearson_coef, p_value = stats.pearsonr(df['horsepower'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P = ", p_value)  

## Length vs. Price 

In [ ]:
pearson_coef, p_value = stats.pearsonr(df['length'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P = ", p_value)  

## Width vs. Price 

In [ ]:
pearson_coef, p_value = stats.pearsonr(df['width'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P =", p_value ) 

## CONCLUSION IMPORTANT VARIABLES
We now have a better idea of what our data looks like and which variables are important to take into account when predicting the car price. We have narrowed it down to the following variables:

Continuous numerical variables:

Length
Width
Curb-weight
Engine-size
Horsepower
City-mpg
Highway-mpg
Wheel-base
Bore
Categorical variables:

Drive-wheels
As we now move into building machine learning models to automate our analysis, feeding the model with variables that meaningfully affect our target variable will improve our model's prediction performance.

## LINEAR REGRESSION & MULTIPLE LINEAR REGRESSION

---

### Simple Linear Regression (SLR)

Models the relationship between **one** predictor $x$ and a response $y$:

$$\hat{y} = b_0 + b_1 x$$

| Symbol | Meaning |
|--------|---------|
| $b_0$ | Intercept — predicted $y$ when $x = 0$ |
| $b_1$ | Slope — change in $y$ per unit increase in $x$ |

**Cars dataset example:**
- $x$ = `engine-size`, $y$ = `price`
- Positive slope → larger engines = higher price

---

### Multiple Linear Regression (MLR)

Extends SLR to **multiple** predictors:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_n x_n$$

**Cars dataset example:**
- $x_1$ = `engine-size`, $x_2$ = `horsepower`, $x_3$ = `highway-mpg`, $x_4$ = `curb-weight`, $y$ = `price`
- Each $b_i$ = effect of that feature **holding all others constant**

---

### Key Metrics

| Metric | Description |
|--------|-------------|
| **R²** | % of variance in $y$ explained by the model (0–1, higher = better) |
| **MSE** | Mean Squared Error — average squared prediction error (lower = better) |
| `model.coef_` | Slope for each feature |
| `model.intercept_` | Baseline predicted value |

**Rule of thumb:** MLR R² > SLR R² → adding features improved the model.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

# Reload the cars dataset to ensure correct df
df_cars = pd.read_csv("usedcars.csv", header=0)

# --- Simple Linear Regression: engine-size → price ---
slr = LinearRegression()
X_slr = df_cars[["engine-size"]]
y_cars = df_cars["price"]

slr.fit(X_slr, y_cars)
y_pred_slr = slr.predict(X_slr)

print("=== Simple Linear Regression ===")
print(f"  Intercept (b0) : {slr.intercept_:.2f}")
print(f"  Slope     (b1) : {slr.coef_[0]:.2f}  (price change per unit engine-size)")
print(f"  R²             : {r2_score(y_cars, y_pred_slr):.4f}")
print(f"  MSE            : {mean_squared_error(y_cars, y_pred_slr):,.2f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_cars["engine-size"], y_cars, alpha=0.5, label="Actual")
axes[0].plot(df_cars["engine-size"], y_pred_slr, color="red", linewidth=2, label="Predicted")
axes[0].set_xlabel("Engine Size")
axes[0].set_ylabel("Price ($)")
axes[0].set_title(f"SLR: Engine Size vs Price  (R²={r2_score(y_cars, y_pred_slr):.3f})")
axes[0].legend()

# --- Multiple Linear Regression ---
features = ["engine-size", "horsepower", "highway-mpg", "curb-weight"]
X_mlr = df_cars[features].dropna()
y_mlr = df_cars.loc[X_mlr.index, "price"]

mlr = LinearRegression()
mlr.fit(X_mlr, y_mlr)
y_pred_mlr = mlr.predict(X_mlr)

print("\n=== Multiple Linear Regression ===")
print(f"  Intercept (b0) : {mlr.intercept_:.2f}")
for feat, coef in zip(features, mlr.coef_):
    print(f"  {feat:<20}: {coef:.4f}")
print(f"  R²             : {r2_score(y_mlr, y_pred_mlr):.4f}")
print(f"  MSE            : {mean_squared_error(y_mlr, y_pred_mlr):,.2f}")

axes[1].scatter(y_mlr, y_pred_mlr, alpha=0.5)
axes[1].plot([y_mlr.min(), y_mlr.max()], [y_mlr.min(), y_mlr.max()], "r--", linewidth=2)
axes[1].set_xlabel("Actual Price ($)")
axes[1].set_ylabel("Predicted Price ($)")
axes[1].set_title(f"MLR: Actual vs Predicted  (R²={r2_score(y_mlr, y_pred_mlr):.3f})")

plt.tight_layout()
plt.show()

## POLYNOMIAL REGRESSION

---

### What is it?

Polynomial regression is an extension of linear regression that models **non-linear relationships** by adding polynomial terms:

$$\hat{y} = b_0 + b_1 x + b_2 x^2 + b_3 x^3 + \cdots + b_n x^n$$

It is still a **linear model** — "linear" refers to the coefficients $b_i$, not to $x$.

---

### When to use it?

Use polynomial regression when:
- A scatter plot shows a **curved** (non-linear) pattern
- A residual plot from SLR shows a systematic curve (not random scatter)

---

### Key concept: degree

| Degree | Shape |
|--------|-------|
| 1 | Straight line (SLR) |
| 2 | Parabola (U or ∩ shape) |
| 3 | S-curve |
| n | More complex curves |

**Warning:** High-degree polynomials can **overfit** — fitting the noise instead of the pattern.

---

### Cars dataset example

- $x$ = `highway-mpg`, $y$ = `price`
- A higher-degree polynomial may capture the non-linear drop in price as mpg increases

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

df_cars = pd.read_csv("usedcars.csv", header=0)

x = df_cars["highway-mpg"]
y = df_cars["price"]

x_range = np.linspace(x.min(), x.max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
degrees = [1, 2, 3]

for ax, deg in zip(axes, degrees):
    poly = PolynomialFeatures(degree=deg)
    X_poly = poly.fit_transform(x.values.reshape(-1, 1))
    X_range_poly = poly.transform(x_range)

    model = LinearRegression()
    model.fit(X_poly, y)
    y_pred = model.predict(X_poly)
    y_curve = model.predict(X_range_poly)

    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)

    ax.scatter(x, y, alpha=0.4, color="steelblue", label="Actual")
    ax.plot(x_range, y_curve, color="red", linewidth=2, label=f"Degree {deg}")
    ax.set_xlabel("Highway MPG")
    ax.set_ylabel("Price ($)")
    ax.set_title(f"Polynomial degree={deg}\nR²={r2:.3f}  MSE={mse:,.0f}")
    ax.legend()

plt.suptitle("Polynomial Regression: Highway MPG vs Price", fontsize=14)
plt.tight_layout()
plt.show()

# Print coefficients for degree=3
poly3 = PolynomialFeatures(degree=3)
X_poly3 = poly3.fit_transform(x.values.reshape(-1, 1))
model3 = LinearRegression().fit(X_poly3, y)

print("=== Polynomial Regression (degree=3) ===")
print(f"  b0 (intercept) : {model3.intercept_:.2f}")
for i, coef in enumerate(model3.coef_[1:], start=1):
    print(f"  b{i} (x^{i})       : {coef:.4f}")
print(f"  R²             : {r2_score(y, model3.predict(X_poly3)):.4f}")
print(f"  MSE            : {mean_squared_error(y, model3.predict(X_poly3)):,.2f}")

## OBSERVATIONS — Polynomial Regression Results

### Model Comparison (highway-mpg → price)

| Degree | R² | MSE | Shape |
|--------|-----|-----|-------|
| 1 — Linear | 0.497 | ~31,635,043 | Straight line |
| 2 — Quadratic | 0.652 | ~21,881,866 | Parabola |
| 3 — Cubic | **0.674** | **~20,474,146** | S-curve |

---

### Key Observations

1. **Degree 1 (Linear) performs poorly** — R² ≈ 0.50 means only half the variance in price is explained. The straight line cannot capture the curved nature of the relationship.

2. **Degree 2 (Quadratic) improves significantly** — R² jumps to 0.65, confirming the relationship is non-linear. As MPG increases, price drops steeply at first, then levels off.

3. **Degree 3 (Cubic) gives the best fit** — R² ≈ 0.67 and lowest MSE. The slight gain over degree 2 suggests a mild inflection in the data at high MPG values.

4. **Diminishing returns beyond degree 3** — Adding more polynomial terms risks **overfitting**: the model memorizes noise instead of learning the true pattern.

---

### Equation (degree=3)

$$\hat{price} = 137923.59 - 8965.43 \cdot x + 204.75 \cdot x^2 - 1.56 \cdot x^3$$

where $x$ = highway MPG. The negative $b_1$ and $b_3$ confirm that price generally decreases as fuel efficiency increases.

## KERNEL DENSITY ESTIMATION 

### What is KDE?

**Kernel Density Estimation (KDE)** is a non-parametric method to estimate the **probability density function** of a continuous variable — essentially a smooth version of a histogram.

Instead of counting values in fixed bins, KDE places a small smooth curve (kernel) at each data point and sums them all:

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\!\left(\frac{x - x_i}{h}\right)$$

| Term | Meaning |
|------|---------|
| $n$ | Number of data points |
| $h$ | **Bandwidth** — controls smoothness (wider = smoother) |
| $K$ | Kernel function (usually Gaussian bell curve) |

---

### When to use it?

- To visualize the **shape of a distribution** without assuming normality
- To **compare distributions** of two or more groups on the same plot
- As an alternative to histograms for continuous data

---

### KDE vs Histogram

| Feature | Histogram | KDE |
|---------|-----------|-----|
| Shape | Blocky (bin-dependent) | Smooth continuous curve |
| Requires bin size | Yes | No (uses bandwidth) |
| Shows density | Approximation | Direct estimate |
| Sensitive to parameters | Bin width | Bandwidth |

---

### In Python (seaborn)

```python
sns.kdeplot(data=df, x="price", fill=True)          # single variable
sns.kdeplot(data=df, x="price", hue="body-style")   # compare groups
```

The area under the KDE curve always equals **1** (it is a proper probability density).

In [ ]:
import numpy as npy
import pandas as pds
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Generating Sample Data
npy.random.seed(42)
x = npy.random.rand(100) * 10
y = 3 * x + npy.random.normal(0, 3, 100)  # Linear relation with noise
data = pds.DataFrame({'X': x, 'Y': y})

# Splitting Data
X_train, X_test, y_train, y_test = train_test_split(data[['X']], data['Y'], test_size=0.2, random_state=42)

# Training a Model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Plotting KDE for Observed vs. Predicted Values
plt.figure(figsize=(8, 5))
sns.kdeplot(y_test, label='Actual', fill=True, color='blue')
sns.kdeplot(y_pred, label='Predicted', fill=True, color='red')
plt.xlabel('Target Variable')
plt.ylabel('Density')
plt.title('KDE Plot of Actual vs. Predicted Values')
plt.legend()
plt.show()

Interpretation of the KDE Plot
Overlap Between Distributions: The two curves have a significant overlap, indicating that the model has captured the general distribution of the actual target values reasonably well. However, the predicted values slightly deviate from the actual values in some regions.

Peak Differences (Mode Shifts): The blue (actual) curve peaks slightly higher than the red curve, meaning that the actual values are more concentrated around certain values. The red (predicted) curve has a second peak, suggesting that the model may be slightly misestimating certain ranges of the target variable.

Spread of the Distributions: The actual values (blue) seem to have a wider spread, indicating more variation in real-world values. The predicted values (red) appear to be narrower, which suggests the model might be slightly underestimating variance (a sign of over-smoothing or bias).

Tails of the Distributions: The tails of the predicted values closely follow the actual values, meaning the model does not generate extreme outliers beyond what was observed in the data. If there was a significant mismatch in the tails, it could indicate that the model struggles with extreme cases.

Conclusion
KDE plots are a powerful visualization tool for assessing the distribution of predicted values compared to actual values in regression analysis. Replacing deprecated distplot with kdeplot() ensures modern and effective visualization in model evaluation workflows.



## MEASURES FOR IN-SAMPLE EVALUATION

---

### What is In-Sample Evaluation?

In-sample evaluation measures how well a model fits the **training data** — the same data it was built on.

> ⚠️ A good in-sample score does **not** guarantee a good model. Always also validate on unseen data.

---

### Key Metrics

#### 1. Mean Squared Error (MSE)

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

- Penalizes large errors more heavily (squared)
- Units are squared (e.g., dollars²)
- **Lower = better**

#### 2. R² — Coefficient of Determination

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- Proportion of variance in $y$ explained by the model
- Range: $(-\infty, 1]$ — for in-sample: usually $[0, 1]$
- **R² = 1** → perfect fit | **R² = 0** → model explains nothing

#### 3. Mean Absolute Error (MAE)

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

- Same units as $y$ (e.g., dollars) — easier to interpret than MSE
- Less sensitive to outliers than MSE
- **Lower = better**

---

### Quick Reference

| Metric | Formula | Ideal | Sensitive to outliers |
|--------|---------|-------|----------------------|
| MSE | avg(error²) | → 0 | Yes (squared) |
| MAE | avg(\|error\|) | → 0 | No |
| R² | 1 - SS_res/SS_tot | → 1 | Moderate |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df_cars = pd.read_csv("usedcars.csv", header=0)

# --- SLR: engine-size → price ---
X = df_cars[["engine-size"]].dropna()
y = df_cars.loc[X.index, "price"]

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2  = r2_score(y, y_pred)

print("=== In-Sample Evaluation: SLR (engine-size → price) ===")
print(f"  R²  : {r2:.4f}   → model explains {r2*100:.1f}% of price variance")
print(f"  MSE : {mse:,.2f}   → avg squared error in $²")
print(f"  MAE : {mae:,.2f}   → avg absolute error in $")
print(f"  RMSE: {np.sqrt(mse):,.2f}   → same units as price ($)")

# Residual plot
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted Price ($)")
axes[0].set_ylabel("Residuals ($)")
axes[0].set_title("Residual Plot — SLR")

axes[1].hist(residuals, bins=20, edgecolor="black", color="steelblue")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Residuals — SLR")

plt.suptitle(f"In-Sample Evaluation  |  R²={r2:.3f}  MSE={mse:,.0f}  MAE={mae:,.0f}", fontsize=13)
plt.tight_layout()
plt.show()

## IBM DEVELOPMENT MODELS 

In [ ]:
file_path= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/automobileEDA.csv"

import urllib.request

urllib.request.urlretrieve(file_path, "usedcars.csv")
file_name="usedcars.csv"


In [ ]:
import pandas as pd

df = pd.read_csv(file_name, header=0)
df.head()

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lm = LinearRegression()
lm

In [ ]:
X = df[['highway-mpg']]
Y = df['price']

In [ ]:
lm.fit(X,Y)

In [ ]:
Yhat=lm.predict(X)
Yhat[0:5]

In [ ]:
lm.intercept_

In [ ]:
lm.coef_

In [ ]:
lm1 = LinearRegression()
lm1
lm1.fit(df[['engine-size']], df[['price']])
lm1

In [ ]:
# Slope 
lm1.coef_

# Intercept
lm1.intercept_

In [ ]:
# using X and Y  
Yhat=-7963.34 + 166.86*X

Price=-7963.34 + 166.86*df['engine-size']

In [ ]:
Z = df[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']]

In [ ]:
lm.fit(Z, df['price'])

In [ ]:
lm.intercept_

In [ ]:
lm.coef_

In [ ]:
lm2 = LinearRegression()
lm2.fit(df[['normalized-losses' , 'highway-mpg']],df['price'])

In [ ]:
lm2.coef_

In [ ]:
import seaborn as sns
%matplotlib inline

## REGRESSION

In [ ]:
width = 12
height = 10
import matplotlib.pyplot as plt
plt.figure(figsize=(width, height))
sns.regplot(x="highway-mpg", y="price", data=df)
plt.ylim(0,)

In [ ]:
plt.figure(figsize=(width, height))
sns.regplot(x="peak-rpm", y="price", data=df)
plt.ylim(0,)

In [ ]:
df[["peak-rpm","highway-mpg","price"]].corr()

## RESIDUAL PLOT

In [ ]:
width = 12
height = 10
plt.figure(figsize=(width, height))
sns.residplot(x=df['highway-mpg'], y=df['price'])
plt.show()

## MULTIPLE LINEAR REGRESSION

In [ ]:
Y_hat = lm.predict(Z)

In [ ]:
plt.figure(figsize=(width, height))


ax1 = sns.distplot(df['price'], hist=False, color="r", label="Actual Value")
sns.distplot(Y_hat, hist=False, color="b", label="Fitted Values" , ax=ax1)


plt.title('Actual vs Fitted Values for Price')
plt.xlabel('Price (in dollars)')
plt.ylabel('Proportion of Cars')

plt.show()
plt.close()

## POLYNOMICAL REGRESSION AND PIPELINES

In [ ]:
def PlotPolly(model, independent_variable, dependent_variabble, Name):
    x_new = np.linspace(15, 55, 100)
    y_new = model(x_new)

    plt.plot(independent_variable, dependent_variabble, '.', x_new, y_new, '-')
    plt.title('Polynomial Fit with Matplotlib for Price ~ Length')
    ax = plt.gca()
    ax.set_facecolor((0.898, 0.898, 0.898))
    fig = plt.gcf()
    plt.xlabel(Name)
    plt.ylabel('Price of Cars')

    plt.show()
    plt.close()

In [ ]:
x = df['highway-mpg']
y = df['price']

In [ ]:
import numpy as np

f = np.polyfit(x, y, 3)
p = np.poly1d(f)
print(p)
p = np.poly1d(f)
print(p)

In [ ]:
PlotPolly(p, x, y, 'highway-mpg')

In [ ]:
np.polyfit(x, y, 3)

In [ ]:
f1 = np.polyfit(x, y, 11)
p1 = np.poly1d(f1)
print(p1)
PlotPolly(p1,x,y, 'Highway MPG')

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
pr=PolynomialFeatures(degree=2)
pr

In [ ]:
Z_pr=pr.fit_transform(Z)

In [ ]:
Z.shape

In [ ]:
Z_pr.shape

## PIPELINE

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
Input=[('scale',StandardScaler()), ('polynomial', PolynomialFeatures(include_bias=False)), ('model',LinearRegression())]

In [ ]:
pipe=Pipeline(Input)
pipe

In [ ]:
Z = Z.astype(float)
pipe.fit(Z,y)

In [ ]:
ypipe=pipe.predict(Z)
ypipe[0:4]

In [ ]:
Input=[('scale',StandardScaler()),('model',LinearRegression())]

pipe=Pipeline(Input)

pipe.fit(Z,y)

ypipe=pipe.predict(Z)
ypipe[0:10]

## R^2 / R-squared Mean Squared Error (MSE) R-squared

R squared, also known as the coefficient of determination, is a measure to indicate how close the data is to the fitted regression line.

The value of the R-squared is the percentage of variation of the response variable (y) that is explained by a linear model.

Mean Squared Error (MSE)

The Mean Squared Error measures the average of the squares of errors. That is, the difference between actual value (y) and the estimated value (ŷ).

## Model 1: Simple Linear Regression 

In [ ]:
#highway_mpg_fit
lm.fit(X, Y)
# Find the R^2
print('The R-square is: ', lm.score(X, Y))

We can say that ~49.659% of the variation of the price is explained by this simple linear model "horsepower_fit".

In [ ]:
Yhat=lm.predict(X)
print('The output of the first four predicted value is: ', Yhat[0:4])

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

class mean_squared_error:
    """
    Mean Squared Error (MSE) loss and gradient.

    This implementation is useful for both evaluation and backpropagation-style
    training workflows.
    """

    def __call__(self, y_true, y_pred):
        return self.forward(y_true, y_pred)

    def forward(self, y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return np.mean(np.square(y_true - y_pred))

    def backward(self, y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        n = y_true.size
        return (2.0 / n) * (y_pred - y_true)

    def compute(self, y_true, y_pred):
        return self.forward(y_true, y_pred)

    def gradient(self, y_true, y_pred):
        return self.backward(y_true, y_pred)
print('The mean square error of price and predicted value is: ', mse)

##  Model 2: Multiple Linear Regression 

In [ ]:
# fit the model 
lm.fit(Z, df['price'])
# Find the R^2
print('The R-square is: ', lm.score(Z, df['price']))


We can say that ~80.896 % of the variation of price is explained by this multiple linear regression "multi_fit"

In [ ]:
Y_predict_multifit = lm.predict(Z)

In [ ]:
import sys
from abc import ABC

class print(ABC):
      def __init__(self, text="", stream=None):
            self.text = text
            self.stream = stream if stream is not None else sys.stdout

      def get_text(self):
            return str(self.text)

      def get_stream(self):
            return self.stream

      def display(self):
            self.get_stream().write(f"{self.get_text()}\n")
            self.get_stream().flush()

      def __str__(self):
            return self.get_text()

      def __repr__(self):
            return f"{self.__class__.__name__}(text={self.get_text()!r})"

## Model 3: Polynomial Fit

In [ ]:
r_squared = r2_score(y, p(x))
print('The R-square value is: ', r_squared)


We can say that ~67.419 % of the variation of price is explained by this polynomial fit.

In [ ]:
class mean_squared_error:

##  Prediction and Decision Making

In [ ]:
new_input=np.arange(1, 100, 1).reshape(-1, 1)

In [ ]:
lm.fit(X, Y)
lm

In [ ]:
yhat=lm.predict(new_input)
yhat[0:5]

In [ ]:
plt.plot(new_input, yhat)
plt.show()

##  Decision Making: Determining a Good Model Fit

Simple Linear Regression: Using Highway-mpg as a Predictor Variable of Price.

R-squared: 0.49659118843391759
MSE: 3.16 x10^7
Multiple Linear Regression: Using Horsepower, Curb-weight, Engine-size, and Highway-mpg as Predictor Variables of Price.

R-squared: 0.80896354913783497
MSE: 1.2 x10^7
Polynomial Fit: Using Highway-mpg as a Predictor Variable of Price.

R-squared: 0.6741946663906514
MSE: 2.05 x 10^7

## MODEL COMPARISON

## Simple Linear Regression Model (SLR) vs Multiple Linear Regression Model (MLR)

Usually, the more variables you have, the better your model is at predicting, but this is not always true. Sometimes you may not have enough data, you may run into numerical problems, or many of the variables may not be useful and even act as noise. As a result, you should always check the MSE and R^2.

In order to compare the results of the MLR vs SLR models, we look at a combination of both the R-squared and MSE to make the best conclusion about the fit of the model.

MSE: The MSE of SLR is 3.16x10^7 while MLR has an MSE of 1.2 x10^7. The MSE of MLR is much smaller.
R-squared: In this case, we can also see that there is a big difference between the R-squared of the SLR and the R-squared of the MLR. The R-squared for the SLR (~0.497) is very small compared to the R-squared for the MLR (~0.809).
This R-squared in combination with the MSE show that MLR seems like the better model fit in this case compared to SLR.

## Simple Linear Model (SLR) vs. Polynomial Fit 

MSE: We can see that Polynomial Fit brought down the MSE, since this MSE is smaller than the one from the SLR.
R-squared: The R-squared for the Polynomial Fit is larger than the R-squared for the SLR, so the Polynomial Fit also brought up the R-squared quite a bit.

## Multiple Linear Regression (MLR) vs. Polynomial Fit

MSE: The MSE for the MLR is smaller than the MSE for the Polynomial Fit.
R-squared: The R-squared for the MLR is also much larger than for the Polynomial Fit.


## Conclusion

Comparing these three models, we conclude that the MLR model is the best model to be able to predict price from our dataset. This result makes sense since we have 27 variables in total and we know that more than one of those variables are potential predictors of the final car price.

## MODEL EVALUATION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df.head()

In [ ]:
df=df._get_numeric_data()
df.head()

In [ ]:
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], errors='ignore')

# Let's take a look at the updated DataFrame
df.head()

## Functions for Plotting 

In [ ]:
def DistributionPlot(RedFunction, BlueFunction, RedName, BlueName, Title):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))
    
    ax1 = sns.kdeplot(RedFunction, color="r", label=RedName)
    ax2 = sns.kdeplot(BlueFunction, color="b", label=BlueName, ax=ax1)

    plt.title(Title)
    plt.xlabel('Price (in dollars)')
    plt.ylabel('Proportion of Cars')
    plt.show()
    plt.close()

In [ ]:
def PollyPlot(xtrain, xtest, y_train, y_test, lr,poly_transform):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))
    
    
    #training data 
    #testing data 
    # lr:  linear regression object 
    #poly_transform:  polynomial transformation object 
 
    xmax=max([xtrain.values.max(), xtest.values.max()])

    xmin=min([xtrain.values.min(), xtest.values.min()])

    x=np.arange(xmin, xmax, 0.1)


    plt.plot(xtrain, y_train, 'ro', label='Training Data')
    plt.plot(xtest, y_test, 'go', label='Test Data')
    plt.plot(x, lr.predict(poly_transform.fit_transform(x.reshape(-1, 1))), label='Predicted Function')
    plt.ylim([-10000, 60000])
    plt.ylabel('Price')
    plt.legend()

## Training and Testing 

In [ ]:
y_data = df['price']

In [ ]:
x_data=df.drop('price',axis=1)


## train_test_split. 

In [ ]:
from sklearn.model_selection import train_test_split


x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.10, random_state=1)


print("number of test samples :", x_test.shape[0])
print("number of training samples:",x_train.shape[0])


In [ ]:
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0) 
print("number of test samples :", x_test1.shape[0])
print("number of training samples:",x_train1.shape[0])

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lre=LinearRegression()

In [ ]:
lre.fit(x_train[['horsepower']], y_train)

In [ ]:
lre.score(x_test[['horsepower']], y_test)

In [ ]:
lre.score(x_train[['horsepower']], y_train)

In [ ]:
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0)
lre.fit(x_train1[['horsepower']],y_train1)
lre.score(x_test1[['horsepower']],y_test1)

## Cross-Validation Score

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
Rcross = cross_val_score(lre, x_data[['horsepower']], y_data, cv=4)

In [ ]:
Rcross

In [ ]:
import sys
from abc import ABC

# Fix the earlier shadowing class in the notebook (cell 53)
# Rename it from `print` to `CustomPrint`

class CustomPrint(ABC):
    def __init__(self, text="", stream=None):
        self.text = text
        self.stream = stream if stream is not None else sys.stdout

    def get_text(self):
        return str(self.text)

    def get_stream(self):
        return self.stream

    def display(self):
        self.get_stream().write(f"{self.get_text()}\n")
        self.get_stream().flush()

    def __str__(self):
        return self.get_text()

    def __repr__(self):
        return f"{self.__class__.__name__}(text={self.get_text()!r})"

In [ ]:
-1 * cross_val_score(lre,x_data[['horsepower']], y_data,cv=4,scoring='neg_mean_squared_error')

In [ ]:
Rc=cross_val_score(lre,x_data[['horsepower']], y_data,cv=2)
Rc.mean()

In [ ]:
from sklearn.model_selection import cross_val_predict

In [ ]:
yhat = cross_val_predict(lre,x_data[['horsepower']], y_data,cv=4)
yhat[0:5]

## Overfitting, Underfitting and Model Selection 

In [ ]:
lr = LinearRegression()
lr.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)

In [ ]:
yhat_train = lr.predict(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_train[0:5]

In [ ]:
yhat_test = lr.predict(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])
yhat_test[0:5]

In [ ]:
Title = 'Distribution  Plot of  Predicted Value Using Training Data vs Training Data Distribution'
DistributionPlot(y_train, yhat_train, "Actual Values (Train)", "Predicted Values (Train)", Title)

In [ ]:
Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'
DistributionPlot(y_test,yhat_test,"Actual Values (Test)","Predicted Values (Test)",Title)

## Overfitting 

Overfitting occurs when the model fits the noise, but not the underlying process. Therefore, when testing your model using the test set, your model does not perform as well since it is modelling noise, not the underlying process that generated the relationship. Let's create a degree 5 polynomial model.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.45, random_state=0)

In [ ]:
pr = PolynomialFeatures(degree=5)
x_train_pr = pr.fit_transform(x_train[['horsepower']])
x_test_pr = pr.fit_transform(x_test[['horsepower']])
pr

In [ ]:
poly = LinearRegression()
poly.fit(x_train_pr, y_train)

In [ ]:
yhat = poly.predict(x_test_pr)
yhat[0:5]

In [ ]:
print("Predicted values:", yhat[0:4])
print("True values:", y_test[0:4].values)

In [ ]:
PollyPlot(x_train['horsepower'], x_test['horsepower'], y_train, y_test, poly,pr)

A polynomial regression model where red dots represent training data, green dots represent test data, and the blue line represents the model prediction.

We see that the estimated function appears to track the data but around 200 horsepower, the function begins to diverge from the data points.

In [ ]:
poly.score(x_train_pr, y_train)

In [ ]:
poly.score(x_test_pr, y_test)

In [ ]:
Rsqu_test = []

order = [1, 2, 3, 4]
for n in order:
    pr = PolynomialFeatures(degree=n)
    
    x_train_pr = pr.fit_transform(x_train[['horsepower']])
    
    x_test_pr = pr.fit_transform(x_test[['horsepower']])    
    
    lr.fit(x_train_pr, y_train)
    
    Rsqu_test.append(lr.score(x_test_pr, y_test))

plt.plot(order, Rsqu_test)
plt.xlabel('order')
plt.ylabel('R^2')
plt.title('R^2 Using Test Data')
plt.text(3, 0.75, 'Maximum R^2 ')    

In [ ]:
def f(order, test_data):
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=test_data, random_state=0)
    pr = PolynomialFeatures(degree=order)
    x_train_pr = pr.fit_transform(x_train[['horsepower']])
    x_test_pr = pr.fit_transform(x_test[['horsepower']])
    poly = LinearRegression()
    poly.fit(x_train_pr,y_train)
    PollyPlot(x_train['horsepower'], x_test['horsepower'], y_train, y_test, poly,pr)

In [ ]:
interact_manual(f, order=(0, 6, 1), test_data=(0.05, 0.95, 0.05))

In [ ]:
pr1=PolynomialFeatures(degree=2)

In [ ]:
x_train_pr1=pr1.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

x_test_pr1=pr1.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

In [ ]:
x_train_pr1.shape #there are now 15 features

In [ ]:
poly1=LinearRegression().fit(x_train_pr1,y_train)

In [ ]:
yhat_test1=poly1.predict(x_test_pr1)

Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'

DistributionPlot(y_test, yhat_test1, "Actual Values (Test)", "Predicted Values (Test)", Title)

The predicted value is higher than actual value for cars where the price $10,000 range, conversely the predicted price is lower than the price cost in the $30,000 to $40,000 range. As such the model is not as accurate in these ranges.

## Ridge Regression



In this section, we will review Ridge Regression and see how the parameter alpha changes the model. Just a note, here our test data will be used as validation data.

In [ ]:
pr=PolynomialFeatures(degree=2)
x_train_pr=pr.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])
x_test_pr=pr.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg','normalized-losses','symboling']])

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
RigeModel=Ridge(alpha=1)

In [ ]:
RigeModel.fit(x_train_pr, y_train)

In [ ]:
yhat = RigeModel.predict(x_test_pr)

In [ ]:
print('predicted:', yhat[0:4])
print('test set :', y_test[0:4].values)

In [ ]:
from tqdm import tqdm

Rsqu_test = []
Rsqu_train = []
dummy1 = []
Alpha = 10 * np.array(range(0,1000))
pbar = tqdm(Alpha)

for alpha in pbar:
    RigeModel = Ridge(alpha=alpha) 
    RigeModel.fit(x_train_pr, y_train)
    test_score, train_score = RigeModel.score(x_test_pr, y_test), RigeModel.score(x_train_pr, y_train)
    
    pbar.set_postfix({"Test Score": test_score, "Train Score": train_score})

    Rsqu_test.append(test_score)
    Rsqu_train.append(train_score)

In [ ]:
width = 12
height = 10
plt.figure(figsize=(width, height))

plt.plot(Alpha,Rsqu_test, label='validation data  ')
plt.plot(Alpha,Rsqu_train, 'r', label='training Data ')
plt.xlabel('alpha')
plt.ylabel('R^2')
plt.legend()

The blue line represents the R^2 of the validation data, and the red line represents the R^2 of the training data. The x-axis represents the different values of Alpha.

Here the model is built and tested on the same data, so the training and test data are the same.

The red line in Figure 4 represents the R^2 of the training data. As alpha increases the R^2 decreases. Therefore, as alpha increases, the model performs worse on the training data

The blue line represents the R^2 on the validation data. As the value for alpha increases, the R^2 increases and converges at a point.

In [ ]:
RigeModel = Ridge(alpha=10) 
RigeModel.fit(x_train_pr, y_train)
RigeModel.score(x_test_pr, y_test)

## Grid Search


The term alpha is a hyperparameter. Sklearn has the class GridSearchCV to make the process of finding the best hyperparameter simpler.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
parameters1= [{'alpha': [0.001,0.1,1, 10, 100, 1000, 10000, 100000, 100000]}]
parameters1

In [ ]:
RR=Ridge()
RR

In [ ]:
Grid1 = GridSearchCV(RR, parameters1,cv=4)

In [ ]:
Grid1.fit(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_data)

In [ ]:
BestRR=Grid1.best_estimator_
BestRR

In [ ]:
BestRR.score(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_test)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_data[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

# Define parameter grid without 'normalize'
parameters2 = [{'alpha': [0.001, 0.1, 1, 10, 100, 1000, 10000, 100000, 1000000]}]

# Perform Grid Search
Grid2 = GridSearchCV(Ridge(), parameters2, cv=4)
Grid2.fit(x_scaled, y_data)

# Best model
best_ridge_model = Grid2.best_estimator_
best_alpha = Grid2.best_params_['alpha']
best_alpha

In [ ]:
RidgeModel100 = Ridge(alpha=100)
RidgeModel100.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)
test_score  = RidgeModel100.score(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_test)
train_score = RidgeModel100.score(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)
print(f"Train R²: {train_score:.4f}")
print(f"Test  R²: {test_score:.4f}")

In [ ]:
RidgeModel0_001 = Ridge(alpha=0.001)
RidgeModel0_001.fit(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)
test_score  = RidgeModel0_001.score(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_test)
train_score = RidgeModel0_001.score(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']], y_train)
print(f"Train R²: {train_score:.4f}")
print(f"Test  R²: {test_score:.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.DataFrame(Grid2.cv_results_)
alphas  = results['param_alpha'].astype(float)
scores  = results['mean_test_score']

plt.figure(figsize=(10, 5))
plt.semilogx(alphas, scores, marker='o')
plt.axvline(best_alpha, color='red', linestyle='--', label=f'Best alpha = {best_alpha}')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Mean CV R²')
plt.title('GridSearchCV: Ridge Alpha vs Cross-Validated R²')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(results[['param_alpha', 'mean_test_score', 'std_test_score']].sort_values('param_alpha'))